In [1]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve
from scipy.ndimage import gaussian_filter1d
from astropy.stats import sigma_clip
from scipy.optimize import curve_fit
import batman

# Configurações globais de plotagem
for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 10

print("="*60)
print("INICIANDO PIPELINE: AU MIC (Planetas b e c)")
print("="*60)

# ============================================================
# PASSO 1 e 2: FUNÇÕES DO MODELO ROTACIONAL (Limpeza da Estrela)
# ============================================================
def gerar_modelo_rotacional_cadencia(t, f, mask_good_flares, cadencia_s=120, janela_horas=10, sigma_clip_val=1.5):
    pontos_por_hora = 3600 / cadencia_s
    janela_pontos = int(janela_horas * pontos_por_hora)
    sigma_pontos = janela_pontos / 8
    
    quebras = list(np.where(np.diff(t) > (20 / (24 * 60)))[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]
    
    modelo_final = np.zeros_like(f)
    
    for i0, i1 in zip(seg_inicios, seg_fins):
        t_seg, f_seg, mask_seg = t[i0:i1], f[i0:i1], mask_good_flares[i0:i1]
        if len(t_seg) < janela_pontos / 4:
            modelo_final[i0:i1] = np.nanmedian(f_seg)
            continue
            
        f_limpo = np.copy(f_seg)
        for _ in range(10):
            temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
            clipped = sigma_clip(f_seg - temp_smooth, sigma_lower=10.0, sigma_upper=sigma_clip_val, maxiters=1, cenfunc='median', stdfunc='mad_std')            
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            f_limpo[~mask_seg] = temp_smooth[~mask_seg] 
            
        modelo_final[i0:i1] = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
    return modelo_final

def ajustar_trecho_especifico(t, f, mask_good_flares, t_inicio, t_fim, cadencia_s=120, janela_horas=5, sigma_upper=2.0, sigma_lower=2.0, iteracoes=6):
    buffer_dias = (janela_horas * 1.5) / 24.0 
    idx_calc = np.where((t >= t_inicio - buffer_dias) & (t <= t_fim + buffer_dias))[0]
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    if len(idx_alvo) == 0: return idx_alvo, np.array([])

    t_calc, f_calc, mask_calc = t[idx_calc], f[idx_calc], mask_good_flares[idx_calc]
    sigma_pontos = (janela_horas * (3600 / cadencia_s)) / 8
    f_limpo = np.copy(f_calc)
    
    for _ in range(iteracoes):
        temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
        clipped = sigma_clip(f_calc - temp_smooth, sigma_lower=sigma_lower, sigma_upper=sigma_upper, maxiters=1, cenfunc='median', stdfunc='mad_std')
        mascara_combinada = (~clipped.mask) & mask_calc
        t_bons, f_bons = t_calc[mascara_combinada], f_limpo[mascara_combinada]
        if len(t_bons) > 2: f_limpo = np.interp(t_calc, t_bons, f_bons)
        else: f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            
    modelo_calc = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
    return idx_alvo, modelo_calc[np.where(idx_calc == idx_alvo[0])[0][0] : np.where(idx_calc == idx_alvo[-1])[0][0] + 1]

def ajustar_trecho_linear(t, f, mask_good_flares, t_inicio, t_fim, sigma_upper=2.0, sigma_lower=3.0, offset_y=0.0, tilt=0.0):
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    if len(idx_alvo) == 0: return idx_alvo, np.array([])

    t_alvo, f_alvo = t[idx_alvo], f[idx_alvo]
    t_limpo, f_limpo = t_alvo[mask_good_flares[idx_alvo]], f_alvo[mask_good_flares[idx_alvo]]
    clipped = sigma_clip(f_limpo, sigma_lower=sigma_lower, sigma_upper=sigma_upper, maxiters=2)
    bons = ~clipped.mask

    if np.sum(bons) > 2:
        coefs = np.polyfit(t_limpo[bons], f_limpo[bons], deg=1)
        t_centro = np.mean(t_alvo)
        modelo_linear = (coefs[0] + tilt) * (t_alvo - t_centro) + np.polyval(coefs, t_centro) + offset_y
    else:
        modelo_linear = (np.ones_like(t_alvo) * np.nanmedian(f_alvo)) + offset_y
    return idx_alvo, modelo_linear

def costurar_bordas(t, modelo, bordas, tamanho_janela=25, sigma_gauss=0):
    modelo_costurado = np.copy(modelo).astype(float)
    for borda_idx in sorted(bordas):
        inicio, fim = max(0, borda_idx - tamanho_janela), min(len(t) - 1, borda_idx + tamanho_janela)
        if (fim - inicio) < 10: continue
        meia_zona = (fim - inicio) // 4
        fim_esq, ini_dir = max(inicio + 2, borda_idx - meia_zona), min(fim - 2, borda_idx + meia_zona)
        idx_esq, idx_dir = np.arange(inicio, fim_esq), np.arange(ini_dir, fim + 1)
        if len(idx_esq) < 5 or len(idx_dir) < 5: continue
            
        t_centro = t[borda_idx]
        poly_esq = np.polyfit(t[idx_esq] - t_centro, modelo_costurado[idx_esq], deg=2)
        poly_dir = np.polyfit(t[idx_dir] - t_centro, modelo_costurado[idx_dir], deg=2)
        
        zona_miolo = np.arange(fim_esq, ini_dir + 1)
        t_miolo = t[zona_miolo] - t_centro
        x_norm = np.clip((t[zona_miolo] - t[fim_esq]) / (t[ini_dir] - t[fim_esq] + 1e-10), 0, 1)
        peso = 3 * x_norm**2 - 2 * x_norm**3 
        modelo_costurado[zona_miolo] = (1 - peso) * np.polyval(poly_esq, t_miolo) + peso * np.polyval(poly_dir, t_miolo)
        
    if sigma_gauss > 0:
        quebras = list(np.where(np.diff(t) > np.median(np.diff(t)) * 5)[0] + 1)
        for i0, i1 in zip([0] + quebras, quebras + [len(t)]):
            if len(modelo_costurado[i0:i1]) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(modelo_costurado[i0:i1], sigma=sigma_gauss)
    return modelo_costurado

def ajustar_trecho_polinomial(t, f, mask_good_flares, t_inicio, t_fim, grau, sigma_upper=2.0, sigma_lower=2.0):
    """
    Ajusta um polinômio local para preservar trânsitos, isolando a região do trânsito 
    e ajustando a linha de base aos pontos fora dele usando sigma clipping.
    """
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    
    # Se a região não estiver nos dados (ex: gap de observação), retorna vazio
    if len(idx_alvo) == 0: 
        return idx_alvo, np.array([])

    t_alvo = t[idx_alvo]
    f_alvo = f[idx_alvo]
    
    # Aplica a máscara inicial de flares
    t_limpo = t_alvo[mask_good_flares[idx_alvo]]
    f_limpo = f_alvo[mask_good_flares[idx_alvo]]
    
    # O sigma_clip inferior (sigma_lower) é vital aqui para ignorar o trânsito 
    # durante o ajuste do polinômio da estrela
    clipped = sigma_clip(f_limpo, sigma_lower=sigma_lower, sigma_upper=sigma_upper, maxiters=2)
    bons = ~clipped.mask

    # Precisamos de pontos suficientes para ajustar o polinômio do grau exigido
    if np.sum(bons) > grau + 1:
        # Centralizamos o tempo para evitar instabilidade numérica na matriz do polinômio
        t_centro = np.mean(t_alvo)
        coefs = np.polyfit(t_limpo[bons] - t_centro, f_limpo[bons], deg=grau)
        modelo_poly = np.polyval(coefs, t_alvo - t_centro)
    else:
        # Fallback de segurança se o clipping remover dados demais
        modelo_poly = np.ones_like(t_alvo) * np.nanmedian(f_alvo)
        
    return idx_alvo, modelo_poly
# --- EXECUÇÃO: DOWNLOAD E LIMPEZA ---
print("\nBuscando e baixando dados de AU Mic do MAST...")
search_result = search_lightcurve("AU Mic")
indices_alvo = [0, 2, 4] # Setores 1, 27 e 95

t_todos, f_todos, modelo_todos, residuo_todos = [], [], [], []

regioes_ajuste_local = [
    [3884.6271, 3884.9503, 10.0, 1.2, 3.0, 12], [3885.9927, 3886.4632, 10.0, 5, 5.0, 12], [3900.1457, 3901.0141, 10.0, 0.8, 0.8, 10],
    [1330.2440, 1330.6343, 8, 1.2, 1.0, 12],[1347.0770, 1347.662, 20, 1.2, 1.0, 10],[2041.013, 2041.4180, 10, 1.2, 1.0, 200],    [3902.9503, 3903.4152, 10.0, 0.8, 0.8, 10],
    [2057.991, 2058.425, 8, 1.2, 1.0, 12],    #[3886.9927, 3886.4709, 8.0, 5, 5.0, 12]
    
   # ,


]
## Ajustes Polinomiais EXCLUSIVOS para proteger os trânsitos (graus 3 ou 4)
regioes_ajuste_polinomial = [
    #[1330.2440, 1330.6343, 3, 2.0, 0.5],
    #[1347.0770, 1347.6620, 3, 2.0, 0.5],
    #[2041.0130, 2041.4180, 4, 2.0, 0.5],
    [2049.4540, 2049.9840, 4, 2.0, 0.5],
    [3886.9927, 3886.4709, 3, 2.0, 0.5]
    #[2057.991, 2058.425, 2, 2.0, 0.5] ,
]

regioes_ajuste_linear = [
    [3906.9180, 3907.1300, 15.5, 9.0, 0.0001, 0.0005]
]

for idx in indices_alvo:
    print(f"A processar Índice de Busca [{idx}]...")
    lc = search_result[idx].download().remove_nans().remove_outliers()
    lc = lc[lc.quality == 0].normalize().remove_nans()

    t_sec = np.ascontiguousarray(lc.time.value, dtype=np.float64)
    f_sec = np.ascontiguousarray(lc.flux, dtype=np.float64)
    
    # Máscara para evitar que o modelo siga flares
    mask_good_flares = (f_sec - gaussian_filter1d(f_sec, sigma=15)) < (3 * np.nanstd(f_sec - gaussian_filter1d(f_sec, sigma=15)))
    
    # Gera o modelo base (onde o erro dos trânsitos ocorria)
    modelo_manchas = gerar_modelo_rotacional_cadencia(t_sec, f_sec, mask_good_flares, cadencia_s=120, janela_horas=10, sigma_clip_val=1.8)
    bordas_indices = set()

    # 1. Aplica Ajustes Locais (Anomalias Gerais da Estrela)
    for ini, fim, janela_h, sig_up, sig_low, iters in regioes_ajuste_local:
        if np.any((t_sec >= ini) & (t_sec <= fim)):
            idx_alvo, mod_local = ajustar_trecho_especifico(t_sec, f_sec, mask_good_flares, ini, fim, 120, janela_h, sig_up, sig_low, iters)
            if len(idx_alvo) > 0:
                modelo_manchas[idx_alvo] = mod_local
                bordas_indices.update([idx_alvo[0], idx_alvo[-1]])

    # 2. Aplica Ajustes Polinomiais (Proteção dos Trânsitos)
    # 2. Aplica Ajustes Polinomiais (Proteção dos Trânsitos)
    for ini, fim, grau, sig_up, sig_low in regioes_ajuste_polinomial:
        if np.any((t_sec >= ini) & (t_sec <= fim)):
            idx_alvo, mod_poly = ajustar_trecho_polinomial(t_sec, f_sec, mask_good_flares, ini, fim, grau, sig_up, sig_low)
            
            if len(idx_alvo) > 30: # Garante que tem tamanho suficiente para suavizar
                # Define quantos pontos vão se misturar nas pontas (aprox 30-40 minutos em cadência de 120s)
                pontos_blend = 15 
                
                mod_antigo = np.copy(modelo_manchas[idx_alvo])
                mod_novo = np.copy(mod_poly)
                
                # Cria rampas de transição suave (de 0 a 1)
                fade_in = np.linspace(0, 1, pontos_blend)
                fade_out = np.linspace(1, 0, pontos_blend)
                
                # Mistura o modelo velho com o novo apenas nas extremidades
                mod_novo[:pontos_blend] = mod_antigo[:pontos_blend] * (1 - fade_in) + mod_novo[:pontos_blend] * fade_in
                mod_novo[-pontos_blend:] = mod_antigo[-pontos_blend:] * (1 - fade_out) + mod_novo[-pontos_blend:] * fade_out
                
                # Aplica o modelo misturado no vetor principal
                modelo_manchas[idx_alvo] = mod_novo
                
                # Como já fizemos a transição perfeita, não precisamos mais mandar o idx_alvo pras bordas_indices
    # 3. Aplica Ajustes Lineares
    for ini, fim, sig_up, sig_low, off_y, tilt_val in regioes_ajuste_linear:
        if np.any((t_sec >= ini) & (t_sec <= fim)):
            idx_alvo, mod_linear = ajustar_trecho_linear(t_sec, f_sec, mask_good_flares, ini, fim, sig_up, sig_low, off_y, tilt_val)
            if len(idx_alvo) > 0:
                modelo_manchas[idx_alvo] = mod_linear
                bordas_indices.update([idx_alvo[0], idx_alvo[-1]])

    # Costura suave das correções com o resto da curva
    if len(bordas_indices) > 0:
        modelo_manchas_suave = costurar_bordas(t_sec, modelo_manchas, sorted(list(bordas_indices)), 25, 3)
    else:
        modelo_manchas_suave = np.copy(modelo_manchas)
        quebras = list(np.where(np.diff(t_sec) > np.median(np.diff(t_sec)) * 5)[0] + 1)
        for i0, i1 in zip([0] + quebras, quebras + [len(t_sec)]):
            if i1 - i0 > 1: modelo_manchas_suave[i0:i1] = gaussian_filter1d(modelo_manchas_suave[i0:i1], sigma=3)

    # Guarda os resultados com os trânsitos preservados
    t_todos.append(t_sec)
    f_todos.append(f_sec)
    modelo_todos.append(modelo_manchas_suave)
    residuo_todos.append(f_sec / modelo_manchas_suave)

# Vetores globais unificados
t = np.concatenate(t_todos)
residual_manchas = np.concatenate(residuo_todos)
#print("Limpeza concluída! Vetor de dados preparado e trânsitos preservados.")

INICIANDO PIPELINE: AU MIC (Planetas b e c)

Buscando e baixando dados de AU Mic do MAST...
A processar Índice de Busca [0]...
A processar Índice de Busca [2]...
A processar Índice de Busca [4]...


In [15]:
# ============================================================
# VISUALIZAÇÃO: CURVA ORIGINAL vs MODELO ROTACIONAL (Setores)
# ============================================================
%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt

print("\n" + "="*60)
print("VISUALIZAÇÃO: DADOS ORIGINAIS vs MODELO ROTACIONAL")
print("="*60)

# Nomes dos setores
setor_nomes = ['Setor 1', 'Setor 27', 'Setor 95']

# Cria figura com 3 subplots (um para cada setor)
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for i, (t_sec, f_sec, modelo_sec) in enumerate(zip(t_todos, f_todos, modelo_todos)):
    ax = axes[i]
    
    # Plot dos dados originais
    ax.plot(t_sec, f_sec, 'k.', ms=2, alpha=0.3, label='Dados Originais (Normalizados)')
    
    # Plot do modelo rotacional
    ax.plot(t_sec, modelo_sec, 'r-', lw=2, alpha=0.8, label='Modelo Rotacional (Manchas Estelares)')
    
    # Formatação
    ax.set_ylabel('Fluxo', fontsize=11, fontweight='bold')
    ax.set_title(f'{setor_nomes[i]} - AU Mic', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, linestyle=':', linewidth=0.7)
    ax.legend(loc='upper right', fontsize=10)
    ax.set_ylim([np.nanmin(f_sec) - 0.001, np.nanmax(f_sec) + 0.001])
    
    # Informações do setor
    info_text = f"Pontos: {len(t_sec)} | T_início: {t_sec[0]:.2f} | T_fim: {t_sec[-1]:.2f} BTJD"
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Último eixo com label X
axes[-1].set_xlabel('Tempo [BTJD]', fontsize=11, fontweight='bold')

plt.suptitle('AU Mic - Qualidade do Ajuste do Modelo Rotacional por Setor', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("✓ Plots dos ajustes gerados com sucesso!\n")

# ============================================================
# ESTATÍSTICAS DO AJUSTE
# ============================================================
print("="*60)
print("ESTATÍSTICAS DOS AJUSTES POR SETOR")
print("="*60)

for i, (nome, t_sec, f_sec, modelo_sec) in enumerate(zip(setor_nomes, t_todos, f_todos, modelo_todos)):
    residuos = f_sec - modelo_sec
    residuos_norm = residuos / np.nanmean(f_sec)
    
    # Cálculo do tempo de observação
    tempo_observacao_dias = t_sec[-1] - t_sec[0]
    tempo_observacao_horas = tempo_observacao_dias * 24
    tempo_observacao_minutos = tempo_observacao_horas * 60
    
    # Cadência média (em minutos)
    dt_medio = np.median(np.diff(t_sec)) * 24 * 60  # em minutos
    
    # Duty cycle (porcentagem de tempo com dados)
    n_gaps = len(np.where(np.diff(t_sec) > 0.01)[0])  # Gaps maiores que ~15 minutos
    
    print(f"\n{'='*70}")
    print(f"  {nome.upper()}")
    print(f"{'='*70}")
    print(f"  📊 INFORMAÇÕES GERAIS:")
    print(f"     • Número de pontos: {len(t_sec):,}")
    print(f"     • Número de gaps de dados: {n_gaps}")
    print(f"     • Cadência média: {dt_medio:.1f} minutos")
    print(f"\n  ⏱️  TEMPO DE OBSERVAÇÃO:")
    print(f"     • Início: {t_sec[0]:.4f} BTJD")
    print(f"     • Fim:    {t_sec[-1]:.4f} BTJD")
    print(f"     • Duração: {tempo_observacao_dias:.2f} dias ({tempo_observacao_horas:.1f} horas, {tempo_observacao_minutos:.0f} minutos)")
    print(f"\n  💫 INFORMAÇÕES DO FLUXO:")
    print(f"     • Fluxo médio: {np.nanmean(f_sec):.6f}")
    print(f"     • Fluxo máx: {np.nanmax(f_sec):.6f}")
    print(f"     • Fluxo mín: {np.nanmin(f_sec):.6f}")
    print(f"     • Amplitude (max-min): {np.nanmax(f_sec) - np.nanmin(f_sec):.6e}")
    print(f"\n  📈 ESTATÍSTICAS DO AJUSTE:")
    print(f"     • RMS dos resíduos: {np.nanstd(residuos):.6e}")
    print(f"     • RMS relativo: {np.nanstd(residuos_norm)*1e6:.1f} ppm")
    print(f"     • Resíduo máx: {np.nanmax(np.abs(residuos)):.6e}")
    print(f"     • Resíduo mín: {np.nanmin(np.abs(residuos)):.6e}")
    print(f"     • Resíduo médio: {np.nanmean(np.abs(residuos)):.6e}")

print(f"\n{'='*70}")


VISUALIZAÇÃO: DADOS ORIGINAIS vs MODELO ROTACIONAL
✓ Plots dos ajustes gerados com sucesso!

ESTATÍSTICAS DOS AJUSTES POR SETOR

  SETOR 1
  📊 INFORMAÇÕES GERAIS:
     • Número de pontos: 17,687
     • Número de gaps de dados: 41
     • Cadência média: 2.0 minutos

  ⏱️  TEMPO DE OBSERVAÇÃO:
     • Início: 1325.9445 BTJD
     • Fim:    1353.0453 BTJD
     • Duração: 27.10 dias (650.4 horas, 39025 minutos)

  💫 INFORMAÇÕES DO FLUXO:
     • Fluxo médio: 1.004857
     • Fluxo máx: 1.051561
     • Fluxo mín: 0.984843
     • Amplitude (max-min): 6.671751e-02

  📈 ESTATÍSTICAS DO AJUSTE:
     • RMS dos resíduos: 1.354310e-03
     • RMS relativo: 1347.8 ppm
     • Resíduo máx: 3.367840e-02
     • Resíduo mín: 0.000000e+00
     • Resíduo médio: 5.586799e-04

  SETOR 27
  📊 INFORMAÇÕES GERAIS:
     • Número de pontos: 16,767
     • Número de gaps de dados: 3
     • Cadência média: 2.0 minutos

  ⏱️  TEMPO DE OBSERVAÇÃO:
     • Início: 2036.2840 BTJD
     • Fim:    2060.6469 BTJD
     • Duração

In [24]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import batman

print("\n" + "="*60)
print("PASSO 3: AJUSTE FOTOMÉTRICO COM DADOS DE LITERATURA (AU Mic b)")
print("="*60)

# ============================================================
# 1. DICIONÁRIO DE TRÂNSITOS CONHECIDOS (Setor 1 e 27)
# ============================================================
# Formato: Época: {'tc': Centro Observado, 'err': Incerteza do Centro}
transitos_conhecidos_b = {
    0:  {'tc': 1330.39046, 'err': 0.00016},
    2:  {'tc': 1347.31646, 'err': 0.00016},
    84: {'tc': 2041.28238, 'err': 0.00026},
    85: {'tc': 2049.74538, 'err': 0.00026},
    86: {'tc': 2058.20838, 'err': 0.00026}
}

# Constantes do Planeta b
T0_b, T0_b_err = 1330.39051, 0.00015
P_b, P_b_err   = 8.463000, 0.000002
RP_b, A_b, INC_b = 0.0526, 19.1, 89.5
LDC = [0.13, 0.58]

def modelo_ajuste_t0(t_janela, t0_livre, per, rp, a, inc):
    p = batman.TransitParams()
    p.t0, p.per, p.rp, p.a, p.inc = t0_livre, per, rp, a, inc
    p.ecc, p.w, p.u, p.limb_dark = 0.0, 90.0, LDC, "quadratic"
    return batman.TransitModel(p, t_janela).light_curve(p)

# ============================================================
# 2. FUNÇÃO MAPEADORA INTELIGENTE (Mistura Teoria com Literatura)
# ============================================================
def mapear_transitos_hibrido(T0, T0_err, P, P_err, t_array, janela_dias=0.25, conhecidos={}):
    n_min = int(np.ceil((t_array.min() - T0) / P))
    n_max = int(np.floor((t_array.max() - T0) / P))
    
    eventos = [] 
    for n in range(n_min, n_max + 1):
        # Se for um dos antigos conhecidos, puxa da lista que você forneceu
        if n in conhecidos:
            tc_esperado = conhecidos[n]['tc']
            tc_err_esperado = conhecidos[n]['err']
            origem = "Literatura (Conhecido)"
        # Se for um trânsito novo do Setor 95, usa a fórmula Teórica
        else:
            tc_esperado = T0 + n * P
            tc_err_esperado = np.sqrt(T0_err**2 + (n * P_err)**2)
            origem = "Teórico (Previsão Setor Novo)"
            
        # Verifica se temos dados do TESS nesse momento exato
        pontos_na_janela = np.sum((t_array >= tc_esperado - janela_dias) & (t_array <= tc_esperado + janela_dias))
        
        if pontos_na_janela > 10:
            eventos.append({
                'epoch': n, 
                'tc_esperado': tc_esperado, 
                'tc_err': tc_err_esperado,
                'origem': origem
            })
    return eventos

# Mapeia onde o planeta b passou nos nossos dados
eventos_b = mapear_transitos_hibrido(T0_b, T0_b_err, P_b, P_b_err, t, 0.25, transitos_conhecidos_b)

# ============================================================
# 3. ROTINA DE AJUSTE E PLOTAGEM (Calculando O-C para novos)
# ============================================================
%matplotlib qt

n_plots = len(eventos_b)
if n_plots > 0:
    print(f"Encontrados {n_plots} trânsitos de AU Mic b com dados!\n")
    fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4.0), sharey=True)
    if n_plots == 1: axes = [axes]

    for i, ev in enumerate(eventos_b):
        epoch = ev['epoch']
        tc_esperado = ev['tc_esperado']
        origem = ev['origem']
        
        # Isola os dados dessa janela temporal
        janela = (t >= tc_esperado - 0.25) & (t <= tc_esperado + 0.25)
        t_f, fluxo_f = t[janela], residual_manchas[janela]
        
        def wrapper_ajuste(t_j, t0_fit):
            return modelo_ajuste_t0(t_j, t0_fit, P_b, RP_b, A_b, INC_b)

        try:
            # Roda o fitting para achar o centro real nos dados
            popt, pcov = curve_fit(wrapper_ajuste, t_f, fluxo_f, p0=[tc_esperado], bounds=(tc_esperado-0.08, tc_esperado+0.08))
            tc_medido, tc_err_medido = popt[0], np.sqrt(pcov[0,0])
        except:
            tc_medido, tc_err_medido = tc_esperado, ev['tc_err']
            
        fluxo_modelo_ajustado = wrapper_ajuste(t_f, tc_medido)
        
        # --- Cálculo do O-C (Para ver os atrasos das curvas novas) ---
        # Calculamos onde o centro TEÓRICO absoluto seria
        tc_puramente_teorico = T0_b + epoch * P_b
        # Diferença do medido para o teórico (em segundos)
        o_c_segundos = (tc_medido - tc_puramente_teorico) * 24 * 3600
        
        print(f"Época {epoch:03d} | Origem Base: {origem}")
        print(f"  -> T_c Esperado: {tc_esperado:.5f}")
        print(f"  -> T_c Medido  : {tc_medido:.5f} ± {tc_err_medido:.5f}")
        print(f"  -> O-C (Atraso): {o_c_segundos:+.1f} segundos\n")
        
        # Gráficos
        ax = axes[i]
        ax.plot(t_f, fluxo_f, 'k.', ms=3, alpha=0.4)
        ax.plot(t_f, fluxo_modelo_ajustado, color='crimson', lw=2.5)
        ax.axvline(tc_medido, color='crimson', ls='--', alpha=0.5)
        ax.errorbar(tc_medido, 1.002, xerr=tc_err_medido, fmt='o', color='red', ms=4)
        
        # Título diferente se for uma curva "Nova" (Setor 95)
        cor_titulo = 'blue' if 'Novo' in origem else 'black'
        ax.set_title(f"Época {epoch}\n$T_c$: {tc_medido:.4f}\nO-C: {o_c_segundos:+.0f}s", fontsize=10, color=cor_titulo)
        ax.set_xlabel('Tempo [BTJD]')
        ax.grid(alpha=0.2)
        if i == 0: ax.set_ylabel('Fluxo Residual')

    plt.suptitle('AU Mic b - Ajuste Fotométrico (Com Prioris de Literatura e Novas Detecções)', fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()


PASSO 3: AJUSTE FOTOMÉTRICO COM DADOS DE LITERATURA (AU Mic b)
Encontrados 7 trânsitos de AU Mic b com dados!

Época 000 | Origem Base: Literatura (Conhecido)
  -> T_c Esperado: 1330.39046
  -> T_c Medido  : 1330.38596 ± 0.00063
  -> O-C (Atraso): -392.7 segundos

Época 002 | Origem Base: Literatura (Conhecido)
  -> T_c Esperado: 1347.31646
  -> T_c Medido  : 1347.31851 ± 0.00095
  -> O-C (Atraso): +173.1 segundos

Época 084 | Origem Base: Literatura (Conhecido)
  -> T_c Esperado: 2041.28238
  -> T_c Medido  : 2041.28174 ± 0.00059
  -> O-C (Atraso): -66.9 segundos

Época 085 | Origem Base: Literatura (Conhecido)
  -> T_c Esperado: 2049.74538
  -> T_c Medido  : 2049.74505 ± 0.00084
  -> O-C (Atraso): -39.8 segundos

Época 086 | Origem Base: Literatura (Conhecido)
  -> T_c Esperado: 2058.20838
  -> T_c Medido  : 2058.17115 ± 0.00136
  -> O-C (Atraso): -3227.6 segundos

Época 302 | Origem Base: Teórico (Previsão Setor Novo)
  -> T_c Esperado: 3886.21651
  -> T_c Medido  : 3886.26528 ± 0.

In [23]:

# ============================================================
# VISUALIZAÇÃO COMPLETA DOS AJUSTES (Modo Interativo Qt)
# ============================================================
%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import batman

print("\n" + "="*60)
print("VISUALIZAÇÃO: QUALIDADE DOS AJUSTES DAS CURVAS DE LUZ")
print("="*60)

# Redefine o wrapper para o ajuste
def modelo_ajuste_t0(t_janela, t0_livre, per, rp, a, inc):
    p = batman.TransitParams()
    p.t0, p.per, p.rp, p.a, p.inc = t0_livre, per, rp, a, inc
    p.ecc, p.w, p.u, p.limb_dark = 0.0, 90.0, LDC, "quadratic"
    return batman.TransitModel(p, t_janela).light_curve(p)
# ============================================================
# 1. VISUALIZAÇÃO DO PLANETA b (COM LINHAS E PONTOS 'k.-')
# ============================================================
print("\n>>> Gerando gráficos de AU Mic b (7 trânsitos)...")

n_plots = len(eventos_b)
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey='row')
axes = axes.flatten()

for i, ev in enumerate(eventos_b):
    epoch = ev['epoch']
    tc_esperado = ev['tc_esperado']
    origem = ev['origem']
    
    # Isola os dados dessa janela
    janela = (t >= tc_esperado - 0.25) & (t <= tc_esperado + 0.25)
    t_f, fluxo_f = t[janela], residual_manchas[janela]
    
    def wrapper_ajuste(t_j, t0_fit):
        return modelo_ajuste_t0(t_j, t0_fit, P_b, RP_b, A_b, INC_b)
    
    try:
        popt, pcov = curve_fit(wrapper_ajuste, t_f, fluxo_f, p0=[tc_esperado], bounds=(tc_esperado-0.08, tc_esperado+0.08))
        tc_medido, tc_err_medido = popt[0], np.sqrt(pcov[0,0])
    except:
        tc_medido, tc_err_medido = tc_esperado, ev['tc_err']
    
    fluxo_modelo_ajustado = wrapper_ajuste(t_f, tc_medido)
    
    ax = axes[i]
    # >>> ALTERAÇÃO AQUI: 'k.-' com lw=1 (linha fina) e ms=5 (tamanho do ponto) <<<
    ax.plot(t_f, fluxo_f, 'k.-', lw=1, ms=5, alpha=0.6, label='Dados (residual)')
    
    ax.plot(t_f, fluxo_modelo_ajustado, color='red', lw=2.5, label='Modelo BATMAN', zorder=5)
    ax.axvline(tc_medido, color='blue', ls='--', alpha=0.6, linewidth=1.5)
    ax.fill_between(t_f, 0.99, 1.01, alpha=0.1, color='gray')
    ax.set_xlabel('Tempo [BTJD]', fontsize=9)
    if i % 4 == 0: ax.set_ylabel('Fluxo Residual', fontsize=9)
    cor = 'green' if 'Novo' in origem else 'black'
    ax.set_title(f"Época {epoch}\n$T_c$={tc_medido:.5f}", fontsize=9, color=cor, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.set_ylim([0.9925, 1.0075])

# Remove o último subplot vazio
fig.delaxes(axes[7])

plt.suptitle('AU Mic b - Ajuste Fotométrico: 7 Trânsitos Observados', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("✓ Gráficos do Planeta b gerados com sucesso!\n")


VISUALIZAÇÃO: QUALIDADE DOS AJUSTES DAS CURVAS DE LUZ

>>> Gerando gráficos de AU Mic b (7 trânsitos)...
✓ Gráficos do Planeta b gerados com sucesso!



In [37]:
# ============================================================
# VISUALIZAÇÃO: CURVAS ORIGINAIS DO TELESCÓPIO (SEM AJUSTE)
# ============================================================
%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt

print("\n" + "="*60)
print("DADOS ORIGINAIS: CONFORME RECEBIDO DO TELESCÓPIO")
print("="*60)

# Nomes dos setores
setor_nomes = ['Setor 1 (2018)', 'Setor 27 (2020)', 'Setor 95(2025)']

# Cria figura com 3 subplots (um para cada setor)
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for i, (t_sec, f_sec) in enumerate(zip(t_todos, f_todos)):
    ax = axes[i]
    
    # Plot APENAS dos dados originais (sem modelo)
    ax.plot(t_sec, f_sec, 'k.-', ms=2, alpha=0.5, label='Dados Originais (Normalizados)')
    
    # ============================================================
    # ADICIONA BARRA DE DIAS OBSERVADOS NA PARTE INFERIOR
    # ============================================================
    # Calcula quais dias foram observados (arredonda para dia inteiro)
    dias_unicos = np.unique(np.floor(t_sec).astype(int))
    dias_totais = int(np.ceil(t_sec[-1])) - int(np.floor(t_sec[0])) + 1
    dias_observados = len(dias_unicos)
    
    # Cria barras para cada dia observado (altura = pequena, na base do gráfico)
    y_min, y_max = ax.get_ylim()
    altura_barra = (y_max - y_min) * 0.02  # 2% da altura do gráfico
    y_barra = y_min + altura_barra * 0.5
    
    for dia in dias_unicos:
        ax.barh(y_barra, 1.0, left=dia - 0.5, height=altura_barra * 0.3, 
                color='lightblue', edgecolor='blue', alpha=0.7, linewidth=0.5)
    
    # Formatação
    ax.set_ylabel('Fluxo Normalizado', fontsize=11, fontweight='bold')
    ax.set_title(f'{setor_nomes[i]} - AU Mic', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, linestyle=':', linewidth=0.7)
    ax.legend(loc='upper right', fontsize=10)
    ax.set_ylim([y_min, y_max])
    
    # Informações do setor - AGORA COM DIAS OBSERVADOS
    info_text = f"Dias observados: {dias_observados}/{dias_totais} | Duração: {t_sec[-1] - t_sec[0]:.2f} dias"
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

# Último eixo com label X
axes[-1].set_xlabel('Tempo [BTJD]', fontsize=11, fontweight='bold')

plt.suptitle('AU Mic TESS - Cobertura Temporal dos Dados', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("✓ Visualização dos dados originais concluída!\n")

# ============================================================
# ESTATÍSTICAS DOS DADOS BRUTOS
# ============================================================
print("="*60)
print("ESTATÍSTICAS DOS DADOS ORIGINAIS")
print("="*60)

for i, (nome, t_sec, f_sec) in enumerate(zip(setor_nomes, t_todos, f_todos)):
    
    # Cálculo do tempo de observação
    tempo_observacao_dias = t_sec[-1] - t_sec[0]
    tempo_observacao_horas = tempo_observacao_dias * 24
    tempo_observacao_minutos = tempo_observacao_horas * 60
    
    # Cadência média (em minutos)
    dt_medio = np.median(np.diff(t_sec)) * 24 * 60
    
    # Gaps
    n_gaps = len(np.where(np.diff(t_sec) > 0.01)[0])
    
    # Dias observados
    dias_unicos = np.unique(np.floor(t_sec).astype(int))
    dias_totais = int(np.ceil(t_sec[-1])) - int(np.floor(t_sec[0])) + 1
    dias_observados = len(dias_unicos)
    duty_cycle = (dias_observados / dias_totais) * 100
    
    # Variabilidade do fluxo bruto
    variabilidade = np.nanstd(f_sec)
    amplitude = np.nanmax(f_sec) - np.nanmin(f_sec)
    
    print(f"\n{'='*70}")
    print(f"  {nome.upper()}")
    print(f"{'='*70}")
    print(f"  📊 INFORMAÇÕES GERAIS:")
    print(f"     • Número de pontos: {len(t_sec):,}")
    print(f"     • Dias observados: {dias_observados}/{dias_totais} ({duty_cycle:.1f}% cobertura)")
    print(f"     • Número de gaps: {n_gaps}")
    print(f"     • Cadência média: {dt_medio:.1f} minutos")
    print(f"\n  ⏱️  INTERVALO TEMPORAL:")
    print(f"     • Início: {t_sec[0]:.4f} BTJD")
    print(f"     • Fim:    {t_sec[-1]:.4f} BTJD")
    print(f"     • Duração: {tempo_observacao_dias:.2f} dias ({tempo_observacao_horas:.1f} horas, {tempo_observacao_minutos:.0f} minutos)")
    print(f"\n  💫 FLUXO BRUTO DO TELESCÓPIO:")
    print(f"     • Fluxo médio: {np.nanmean(f_sec):.6f}")
    print(f"     • Fluxo máx: {np.nanmax(f_sec):.6f}")
    print(f"     • Fluxo mín: {np.nanmin(f_sec):.6f}")
    print(f"     • Amplitude (max-min): {amplitude:.6e}")
    print(f"     • Desvio padrão: {variabilidade:.6e}")
    print(f"     • Variabilidade relativa: {variabilidade*1e6:.1f} ppm")

print(f"\n{'='*70}")


DADOS ORIGINAIS: CONFORME RECEBIDO DO TELESCÓPIO
✓ Visualização dos dados originais concluída!

ESTATÍSTICAS DOS DADOS ORIGINAIS

  SETOR 1 (2018)
  📊 INFORMAÇÕES GERAIS:
     • Número de pontos: 17,687
     • Dias observados: 29/30 (96.7% cobertura)
     • Número de gaps: 41
     • Cadência média: 2.0 minutos

  ⏱️  INTERVALO TEMPORAL:
     • Início: 1325.9445 BTJD
     • Fim:    1353.0453 BTJD
     • Duração: 27.10 dias (650.4 horas, 39025 minutos)

  💫 FLUXO BRUTO DO TELESCÓPIO:
     • Fluxo médio: 1.004857
     • Fluxo máx: 1.051561
     • Fluxo mín: 0.984843
     • Amplitude (max-min): 6.671751e-02
     • Desvio padrão: 1.430864e-02
     • Variabilidade relativa: 14308.6 ppm

  SETOR 27 (2020)
  📊 INFORMAÇÕES GERAIS:
     • Número de pontos: 16,767
     • Dias observados: 25/26 (96.2% cobertura)
     • Número de gaps: 3
     • Cadência média: 2.0 minutos

  ⏱️  INTERVALO TEMPORAL:
     • Início: 2036.2840 BTJD
     • Fim:    2060.6469 BTJD
     • Duração: 24.36 dias (584.7 horas,